# Fastest time solution

Import libraries

In [ ]:
import os
import folium
import json
import numpy as np
import branca.colormap as cm

In [ ]:
from src.matrix_maker import create_time_matrix, create_distance_matrix, add_dummy_nodes 
from src.tsp import solve_tsp

Custom colours, use default colourmaps instead

In [ ]:
from pyush import MyColours
cmap = MyColours().specified_colour_ramp(11).hex[::-1]

In [ ]:
# from matplotlib.cm import plasma
# cmap = plasma

## Find the solution

Setting model for cyclist speed, given assumptions, default parameters are:
```
power_output=150              # watts 
weight=70                     # Of cyclist in kg
bike_weight=10                # weight of bike in kg 
drive_train_loss=0.02         # loss from drive train, 0 = no loss, 1 = total loss
area=0.509                    # Front area of cyclist m^2
air_density=1.22601           # kg/m^3
cda=0.4                       # coef of air drag
coef_rolling_resistance=0.005 # coef of rolling risistance

max_speed=45                  # maximum speed of rider in km/hr - 
                              # stops huge speeds downhill
                              
min_speed=10                  # minimum speed of rider in km/hr - 
                              # stops crawling up hills
```

In [ ]:
cyclist_params = {
    "power_output": 100,
    "weight": 60,
    "bike_weight": 12,
    "max_speed": 40,
    "min_speed": 10
}

Builds the time matrix, adding a dummy node for begining/ending point

In [ ]:
locs_time, mat_time = create_time_matrix(**cyclist_params)
locs_dist, mat_dist = create_distance_matrix(**cyclist_params)
dummy_locs, dummy_mat = add_dummy_nodes(locs_time, mat_time)

In [ ]:
solution_idx = solve_tsp(dummy_mat, 0)

Print out of route

In [ ]:
prev_idx = 0
tm = 0
dist = 0

print_str = "Location\t\t Cum. Time\t Cum. Dist\n"
print_str += "------------------------------------------------\n"

for i in solution_idx['route'][1:-1]:
    spc = "\t\t" if len(dummy_locs[i]) > 7 else "\t\t\t"
    print_str += dummy_locs[i].replace("_", " ").capitalize() + spc
    if prev_idx == 0:
        print_str += "0min\t\t 0km\n"
    else:
        tm += round(mat_time[prev_idx-1][i-1]/60)
        dist += round(mat_dist[prev_idx-1][i-1]/1000)
        print_str += f"{tm}min\t\t"
        print_str += f"{dist}km\n"

    prev_idx = i
print(print_str)    

In [ ]:
dist / (tm / 60)

## Map solution

In [ ]:
cafe_locs = []
route = []

for p1, p2 in zip(solution_idx['route'][:-1], solution_idx['route'][1:]):
    if p1 != 0 and p2 != 0:
        loc1 = dummy_locs[p1]
        loc2 = dummy_locs[p2]
        file_name = "routes/" + loc1 + "-" + loc2 + ".json"
        file_name_rev = "routes/" + loc2 + "-" + loc1 + ".json"
        if os.path.exists(file_name):
            # Read the file and get points
            with open(file_name, "r") as f:
                data = json.load(f)
            route.append(data["paths"][0]["points"]["coordinates"])
            cafe_locs.append(data["paths"][0]["points"]["coordinates"][0])
        elif os.path.exists(file_name_rev):
            with open(file_name_rev, "r") as f:
                data = json.load(f)
            route.append(data["paths"][0]["points"]["coordinates"][::-1])
            cafe_locs.append(data["paths"][0]["points"]["coordinates"][-1])
        else:
            print("missing:" + loc1 + "-" + loc2)  

route = np.vstack(route)
# Add last cafe point
cafe_locs.append(route[-1])  

In [ ]:
spacer = 3

route_points = route[..., :2][::spacer, ::-1]
elev = route[::spacer, 2]

In [ ]:
# Create the base map
my_map = folium.Map(location=route_points[0], zoom_start=12, tiles="OpenStreetMap")

# Create a colormap (e.g., Linear from Green -> Yellow -> Red)
colormap = cm.LinearColormap(cmap, vmin=0, vmax=350)

# Draw the route segment by segment
for i in range(len(elev) - 1):
    point_a = route_points[i]
    point_b = route_points[i+1]
    
    # Coordinates for the segment
    segment_coords = [point_a, point_b]
    
    # Calculate average elevation for this segment to determine color
    avg_elevation = (elev[i] + elev[i+1]) / 2
    segment_color = colormap(avg_elevation)
    
    # Add segment to map
    folium.PolyLine(
        locations=segment_coords, 
        color=segment_color, 
        weight=5, 
        opacity=0.9,
        popup=f"Avg Elev: {int(avg_elevation)}m"
    ).add_to(my_map)

# Add Cafe locations
for loc, name in zip(cafe_locs, solution_idx['route'][1:-1]):
    cl = loc[:2][::-1]
    folium.Marker(
        cl, 
        popup=dummy_locs[name].replace("_", " ").capitalize(),
        icon=folium.Icon(color='gray', icon="coffee", prefix="fa")
    ).add_to(my_map)

# Add the colormap legend to the map
colormap.caption = 'Elevation (meters)'
colormap.add_to(my_map)

# Save map
my_map.save("elevation_map.html")